# Full Fine-Tuning All Models - Colab/Drive Runner

Este notebook esta pensado para ejecutarse en Google Colab con GPU usando la carpeta `google_drive_full_finetuning_pack` subida a Drive.

Propiedades importantes:

- Reanuda automaticamente desde `last.pt` si Colab se cae.
- Salta jobs terminados si existe `done.json`.
- Guarda `best.pt`, `last.pt`, `history.csv`, `progress.json`, `metrics.json`, predicciones y matrices de confusion.
- Permite cambiar de cuenta de Google: basta con volver a subir/copiar la carpeta completa y ajustar `PACK_DIR`.
- Ejecuta full fine-tuning por defecto (`freeze_mode = full`).


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json, time

drive.mount('/content/drive')
print('Python:', sys.executable)


## 1. Localizar carpeta del paquete

Si subiste la carpeta completa a `Mi unidad`, normalmente sera:

`/content/drive/MyDrive/google_drive_full_finetuning_pack`

Si la pusiste en otro lugar, edita `PACK_DIR` manualmente.


In [ ]:
# Cambia esto si tu carpeta esta en otra ruta.
PACK_DIR = Path('/content/drive/MyDrive/google_drive_full_finetuning_pack')

# Busqueda automatica si la ruta anterior no existe.
if not PACK_DIR.exists():
    matches = list(Path('/content/drive/MyDrive').rglob('google_drive_full_finetuning_pack'))
    if matches:
        PACK_DIR = matches[0]

assert PACK_DIR.exists(), f'No existe PACK_DIR: {PACK_DIR}'
SCRIPT = PACK_DIR / 'scripts' / 'full_finetune_colab.py'
CONFIG = PACK_DIR / 'configs' / 'full_finetune_config.json'
assert SCRIPT.exists(), SCRIPT
assert CONFIG.exists(), CONFIG
print('PACK_DIR =', PACK_DIR)
print('SCRIPT =', SCRIPT)
print('CONFIG =', CONFIG)


## 2. Verificar GPU y datasets

Esta celda no entrena. Solo confirma que Colab ve la GPU y que los CSV/datasets estan en la carpeta.


In [ ]:
import torch, pandas as pd
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

summary = pd.read_csv(PACK_DIR / 'splits' / 'split_summary.csv')
display(summary)
print('Datasets folder exists:', (PACK_DIR / 'datasets').exists())
print('Results folder:', PACK_DIR / 'results')


## 3. Configuracion de ejecucion

Para correr todo, deja `RUN_DATASETS` y `RUN_MODELS` vacios.

Para probar primero, usa por ejemplo:

```python
RUN_DATASETS = 'brain_tumor_mri_44c'
RUN_MODELS = 'efficientnet_b0'
MAX_JOBS = 1
```


In [ ]:
RUN_DATASETS = ''   # vacio = todos los datasets del config
RUN_MODELS = ''     # vacio = todos los modelos enabled=true del config
MAX_JOBS = 0        # 0 = sin limite; util para probar con 1 o 2 jobs
OVERRIDE_EPOCHS = 0 # 0 = usar config; para smoke test pon 1 o 2
OVERRIDE_BATCH_SIZE = 0 # 0 = usar batch por modelo
NUM_WORKERS = 2
DEVICE = 'auto'
STOP_ON_ERROR = False
FORCE_RERUN_DONE = False
RESTART_CURRENT_JOB = False

print({
    'RUN_DATASETS': RUN_DATASETS or 'ALL',
    'RUN_MODELS': RUN_MODELS or 'ALL_ENABLED',
    'MAX_JOBS': MAX_JOBS,
    'OVERRIDE_EPOCHS': OVERRIDE_EPOCHS,
    'OVERRIDE_BATCH_SIZE': OVERRIDE_BATCH_SIZE,
})


## 4. Dry-run: ver jobs que se ejecutaran

Recomendado antes de dejarlo corriendo.


In [ ]:
cmd = [
    sys.executable, str(SCRIPT), 'run',
    '--pack-dir', str(PACK_DIR),
    '--config', str(CONFIG),
    '--device', DEVICE,
    '--num-workers', str(NUM_WORKERS),
    '--dry-run',
]
if RUN_DATASETS:
    cmd += ['--datasets', RUN_DATASETS]
if RUN_MODELS:
    cmd += ['--models', RUN_MODELS]
if MAX_JOBS:
    cmd += ['--max-jobs', str(MAX_JOBS)]
if OVERRIDE_EPOCHS:
    cmd += ['--epochs', str(OVERRIDE_EPOCHS)]
if OVERRIDE_BATCH_SIZE:
    cmd += ['--batch-size', str(OVERRIDE_BATCH_SIZE)]
print(' '.join(map(str, cmd)))
subprocess.run(cmd, check=True)


## 5. Ejecutar full fine-tuning reanudable

Puedes cortar la ejecucion o cambiar de cuenta. Al volver a correr esta celda, el script:

- salta jobs con `done.json`;
- reanuda jobs con `last.pt`;
- continua jobs interrumpidos desde el ultimo epoch guardado.


In [ ]:
cmd = [
    sys.executable, str(SCRIPT), 'run',
    '--pack-dir', str(PACK_DIR),
    '--config', str(CONFIG),
    '--device', DEVICE,
    '--num-workers', str(NUM_WORKERS),
]
if RUN_DATASETS:
    cmd += ['--datasets', RUN_DATASETS]
if RUN_MODELS:
    cmd += ['--models', RUN_MODELS]
if MAX_JOBS:
    cmd += ['--max-jobs', str(MAX_JOBS)]
if OVERRIDE_EPOCHS:
    cmd += ['--epochs', str(OVERRIDE_EPOCHS)]
if OVERRIDE_BATCH_SIZE:
    cmd += ['--batch-size', str(OVERRIDE_BATCH_SIZE)]
if FORCE_RERUN_DONE:
    cmd += ['--force']
if RESTART_CURRENT_JOB:
    cmd += ['--restart']
if STOP_ON_ERROR:
    cmd += ['--stop-on-error']

print('START:', time.strftime('%Y-%m-%d %H:%M:%S'))
print(' '.join(map(str, cmd)))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print('END:', time.strftime('%Y-%m-%d %H:%M:%S'), 'rc=', rc)
if rc != 0:
    raise RuntimeError(f'Fine-tuning runner failed with rc={rc}')


## 6. Recolectar resultados parciales/finales

Puedes correr esta celda aunque no todo haya terminado.


In [ ]:
cmd = [sys.executable, str(SCRIPT), 'collect', '--pack-dir', str(PACK_DIR)]
subprocess.run(cmd, check=True)
results_csv = PACK_DIR / 'results' / 'ALL_FINETUNE_RESULTS.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df.sort_values(['dataset_id', 'test_macro_f1'], ascending=[True, False]) if not df.empty else df)
else:
    print('No results CSV yet')


## 7. Ver progreso de jobs incompletos


In [ ]:
progress_files = list((PACK_DIR / 'results').rglob('progress.json'))
rows = []
for p in progress_files:
    try:
        data = json.loads(p.read_text())
        data['path'] = str(p)
        rows.append(data)
    except Exception as e:
        rows.append({'path': str(p), 'error': str(e)})
if rows:
    display(pd.DataFrame(rows).sort_values('updated_at', ascending=False))
else:
    print('No progress files yet')
